# Create Swift Format Dataset

Supports nested folder structure:
```
data/
├── documents/
│   ├── aus_passport/
│   ├── aus_medicare_card/{green,blue,yellow}/
│   └── aus_driver_license/{act,nsw,...}/
└── labels/  (mirrors documents/)
```
`template.json` / `template.png` are automatically ignored.

## 1 · Install

In [ ]:
!pip install pypdfium2==4.30.1 pandas Pillow tqdm --quiet
print("Ready")

## 2 · Configuration

In [ ]:
from pathlib import Path

DATA_DIR   = Path("./data") # root data dir
DOC_DIR    = DATA_DIR / "documents" # Store img/pdf
LABEL_DIR  = DATA_DIR / "labels" # 
OUTPUT_DIR = DATA_DIR / "swift_dataset_bill"
SCHEMA_DIR = Path("./schemas")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SKIP_DIRS = {
    "aus_passport",
    "aus_medicare_card",
    "aus_driver_license",
    "aus_wwc_card"
}

MIN_FOR_SPLIT = 1 # minimum number of records for a stem to be split into train/dev/test
DEV_RATIO     = 0.0 # dev set 
TEST_RATIO    = 0.3 # test set
RANDOM_SEED   = 42

## 3 · Load Data

In [ ]:
import json, mimetypes, math, random, io
from collections import defaultdict, Counter
from tqdm import tqdm
import pandas as pd
from PIL import Image
import pypdfium2 as pdfium
import json, mimetypes, math, random, io
from collections import defaultdict, Counter
from tqdm import tqdm
import pandas as pd
from PIL import Image
import pypdfium2 as pdfium

def load_local_data(doc_dir, label_dir, skip_dirs):
    records = []
    all_jsons = [
        p for p in sorted(Path(label_dir).rglob("*.json"))
        if p.stem != "template"
        and not any(d in p.parts for d in skip_dirs)
    ]
    print(f"Found {len(all_jsons)} label files")

    for label_path in tqdm(all_jsons, desc="Loading"):
        rel = label_path.relative_to(label_dir)
        img_path = next(
            (Path(doc_dir) / rel.parent / (label_path.stem + ext)
             for ext in [".jpg",".jpeg",".png",".JPG",".PNG",".pdf"]
             if (Path(doc_dir) / rel.parent / (label_path.stem + ext)).exists()),
            None
        )
        if not img_path:
            print(f"  SKIP (no image): {rel}"); continue

        records.append({
            "filename"   : img_path.name,
            "rel_path"   : str(rel),
            "doc_type"   : "/".join(rel.parts[:-1]),
            "filetype"   : mimetypes.guess_type(img_path)[0] or "image/jpeg",
            "target_data": json.dumps(json.load(open(label_path, encoding="utf-8")), ensure_ascii=False),
            "doc_bytes"  : img_path.read_bytes(),
        })

    print(f"Loaded {len(records)} records")
    return records

all_records = load_local_data(DOC_DIR, LABEL_DIR, SKIP_DIRS)

In [ ]:
df_info = pd.DataFrame([{k:v for k,v in r.items() if k != "doc_bytes"} for r in all_records])
dist = df_info["doc_type"].value_counts().sort_index()
print(dist.to_string())
print(f"\nTotal: {len(df_info)} samples, {len(dist)} types")

## 4 · Split (80 / 0 / 20)

In [ ]:
random.seed(RANDOM_SEED)

by_type = defaultdict(list)
for r in all_records:
    by_type[r["doc_type"]].append(r)

train_records, dev_records, test_records = [], [], []

print(f"{'Document Type':<42} {'N':>5} {'Train':>6} {'Dev':>5} {'Test':>5}")
print("-" * 65)

for doc_type in sorted(by_type):
    recs = by_type[doc_type]
    random.shuffle(recs)
    n = len(recs)

    if n >= MIN_FOR_SPLIT:
        # n_test  = max(1, int(n * TEST_RATIO))
        # n_dev   = max(1, int(n * DEV_RATIO))
        n_test  =  int(n * TEST_RATIO)
        n_dev   =  int(n * DEV_RATIO)
        n_train = n - n_dev - n_test
    else:
        n_train, n_dev, n_test = n, 0, 0

    train_records.extend(recs[:n_train])
    dev_records.extend(recs[n_train:n_train+n_dev])
    test_records.extend(recs[n_train+n_dev:])

    note = "  *" if n < MIN_FOR_SPLIT else ""
    print(f"{doc_type:<42} {n:>5} {n_train:>6} {n_dev:>5} {n_test:>5}{note}")

print("-" * 65)
print(f"{'TOTAL':<42} {len(all_records):>5} {len(train_records):>6} {len(dev_records):>5} {len(test_records):>5}")
print("  * insufficient samples — kept in train only")

## 5 · Extract Images & Build Swift Format

In [ ]:
import json
from pathlib import Path

def build_schema(d):
    return {k: (build_schema(v) if isinstance(v, dict) else ([] if isinstance(v, list) else None))
            for k, v in d.items()}

def load_schema_as_template(schema_path):
    raw   = json.load(open(schema_path))
    props = raw.get("properties", {})
    return {k: "..." for k in props}

# Load schemas
SCHEMA_DIR = Path("/teamspace/studios/this_studio/ocr/sample-for-multi-modal-document-to-json-with-sagemaker-ai/schemas")
SCHEMAS = {}
SCHEMAS["AUS_ELECTRICITY_BILL"] = load_schema_as_template(SCHEMA_DIR / "aus_energy_bill_schema.json")
print("Schema keys:", list(SCHEMAS["AUS_ELECTRICITY_BILL"].keys()))

def extract_images(record, max_pages=2):
    images_dir = OUTPUT_DIR / "images"
    images_dir.mkdir(exist_ok=True)
    stem, filetype = Path(record["filename"]).stem, record["filetype"]
    paths = []
    try:
        if filetype == "application/pdf":
            pdf = pdfium.PdfDocument(record["doc_bytes"])
            for i in range(min(len(pdf), max_pages)):
                p = images_dir / f"{stem}_page{i:03}.png"
                if not p.exists(): pdf[i].render(scale=1.53).to_pil().save(p)
                paths.append(str(p.resolve()))
        else:
            p = images_dir / f"{stem}.png"
            if not p.exists(): Image.open(io.BytesIO(record["doc_bytes"])).save(p)
            paths.append(str(p.resolve()))
    except Exception as e:
        print(f"Error {stem}: {e}")
    return paths

HINTS = {
    "AUS_ELECTRICITY_BILL": (
        "provider_name: exact trading name as printed (e.g. 'AGL', 'Synergy'). "
        "nmi: 10-11 digit number labeled 'NMI' or 'Meter Identifier'. "
        "service_address: supply/meter address — NOT mailing or billing address. "
        "bill_issue_date: date bill was created, NOT the due date. "
        "total_electricity_kwh: null if not explicitly printed as total kWh. "
        "total_excl_gst / total_gst: null if not shown separately. "
        "amount_due: FINAL payable amount after credits — may be negative if in credit. "
        "All monetary values: float, no $ symbol, no commas. "
        "All dates: YYYY-MM-DD format."
    ),
}
DEFAULT_HINT = "Extract all fields carefully. Return null for any field not visible or missing."

def make_swift_sample(record):
    image_paths = extract_images(record)
    if not image_paths: return None

    label_dict = json.loads(record["target_data"])
    dt         = label_dict.get("document_type", "")
    schema     = SCHEMAS.get(dt, build_schema(label_dict))

    user_content = (
        f"Document: {'<image>' * len(image_paths)}\n"
        f"Task: Extract all fields from this {dt} document as a single JSON object.\n"
        f"\n"
        f"Rules:\n"
        f"- Return ONLY valid JSON, no markdown, no explanation\n"
        f"- Use null for any field not visible in the document\n"
        f"- Do NOT omit any key\n"
        f"- Dates must be YYYY-MM-DD format\n"
        f"- Monetary amounts are float (NO commas, NO $ symbol)\n"
        f"- {HINTS.get(dt, DEFAULT_HINT)}\n"
        f"\n"
        f"IMPORTANT: The JSON structure below is a LABEL TEMPLATE only.\n"
        f"Do NOT copy these values. Extract real values from the document image above.\n"
        f"\n"
        f"Output must match this JSON structure (keys and nesting only):\n"
        f"{json.dumps(schema, ensure_ascii=False, indent=2)}"
    )

    return {
        "messages": [
            {"role": "system",    "content": "You are a document processing expert skilled in extracting information from official documents."},
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": json.dumps(label_dict, ensure_ascii=False)},
        ],
        "images": image_paths,
    }

# Preview field keys per document type
keys_by_doctype = defaultdict(set)
for r in all_records:
    label = json.loads(r["target_data"])
    keys_by_doctype[label.get("document_type", "UNKNOWN")].update(label.keys())
for dtype, keys in sorted(keys_by_doctype.items()):
    print(f"  {dtype}: {sorted(keys)}")

## ~~6 · Load Mapping & Schemas~~ (disabled — using raw labels)

In [ ]:
# ── Section disabled: no mapping used ──
# mapping_path = SCHEMA_DIR / "label_to_schema_mapping.json"
# label_to_schema = {}
# 
# if mapping_path.exists():
#     label_to_schema = json.load(open(mapping_path))
#     print("Field mapping:")
#     for dtype, mapping in sorted(label_to_schema.items()):
#         print(f"  {dtype}:")
#         for src, tgt in mapping.items():
#             if not src.startswith("_"):
#                 tgt_str = tgt["_target"] if isinstance(tgt, dict) else (tgt or "(ignored)")
#                 print(f"    {src:<35} → {tgt_str}")
# else:
#     print(f"Warning: {mapping_path} not found")

In [ ]:
# ── Section disabled: no schemas used ──
# schemas = {}
# if SCHEMA_DIR.exists():
#     for p in sorted(SCHEMA_DIR.glob("*.json")):
#         if p.stem == "label_to_schema_mapping": continue
#         schema = json.load(open(p))
#         dtype  = p.stem.replace("_schema","").upper()
#         schemas[dtype] = schema
#         props    = schema.get("properties", {})
#         required = [k for k,v in props.items() if v.get("required") is True]
#         optional = [k for k,v in props.items() if v.get("required") is False]
#         print(f"  {dtype}: {len(required)} required, {len(optional)} optional")
#     print(f"\nLoaded {len(schemas)} schemas")
# else:
#     print(f"Warning: {SCHEMA_DIR} not found")

## 7 · Save Swift Format Files

In [ ]:
def save_swift_file(records, split_name):
    samples, skipped = [], 0
    for r in tqdm(records, desc=split_name):
        s = make_swift_sample(r)
        if s: samples.append(s)
        else: skipped += 1

    out = OUTPUT_DIR / f"conversations_{split_name}_swift_format.json"
    json.dump(samples, open(out,"w",encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"  {split_name}: {len(samples)} samples → {out.name}" +
          (f"  ({skipped} skipped)" if skipped else ""))
    return out

swift_train = save_swift_file(train_records, "train")
swift_dev   = save_swift_file(dev_records,   "dev")
swift_test  = save_swift_file(test_records,  "test")


## 8 · Verify

In [ ]:
def show_distribution(path, name):
    data  = json.load(open(path))
    types = [json.loads(s["messages"][2]["content"]).get("document_type","?") for s in data]
    print(f"{name} ({len(data)} samples):")
    for t, c in sorted(Counter(types).items()):
        print(f"  {t}: {c}")

show_distribution(swift_train, "TRAIN")
print()
show_distribution(swift_dev, "DEV")

In [ ]:
train_data = json.load(open(swift_train))
if train_data:
    s = train_data[0]
    print("Images:", s["images"])
    print("\nResponse (first 300 chars):")
    print(s["messages"][2]["content"][:300])

In [ ]:
print("Output files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name:<50} {f.stat().st_size/1024:>8.1f} KB")
n_img = len(list((OUTPUT_DIR/"images").glob("*")))
print(f"\nImages: {n_img} files")

## Done

### Training config
```python
train_dataset = "./data/swift_dataset/conversations_train_swift_format.json"
val_dataset   = "./data/swift_dataset/conversations_dev_swift_format.json"
test_dataset  = "./data/swift_dataset/conversations_test_swift_format.json"
```

### Adding a new document type
1. Add images → `documents/<type>/<subtype>/`
2. Add labels → `labels/<type>/<subtype>/`
3. Add schema + update `label_to_schema_mapping.json`
4. Re-run notebook